In [9]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec  #grid pels subplots
from matplotlib import colors         #colors
from matplotlib.colors import LogNorm   #normalitza a 0-1 en escala logarítmica 
from matplotlib import patches     #figures
from mpl_toolkits.mplot3d import Axes3D   #eixos en 3D
from mpl_toolkits.axes_grid1 import make_axes_locatable  #per canviar la posició dels eixos
from matplotlib.ticker import NullFormatter   #marques(tics) sense etiquetes en els eixos

from astroML.density_estimation import XDGMM
from astroML.plotting.tools import draw_ellipse
from astroML.crossmatch import crossmatch
from astroML.datasets import fetch_sdss_S82standards, fetch_imaging_sample
from astroML.stats import sigmaG
from astroML.utils.decorators import pickle_results
import os
import pickle

import astropy.table     #paquet per manejar taules de dades
from astropy.table import Table, Column, MaskedColumn   #importa taules, columnes i columnes que emmascaren dades invàlides
from astropy.visualization import astropy_mpl_style  #visualització 
from scipy.stats import gaussian_kde  #representation of a kernel-density estimate using Gaussian kernels.
import seaborn as sns  #llibreria per fer gràfics estadístics
import os.path   #per implementar diferents funcions amb pathnames ("dreceres")

from time import time   #mòdul de funcions de time access 
from sklearn import manifold, datasets #manifold: algoritme de dimensionality reduction 
                                       #sklearn (sci-kit learn): llibreria de Python per machine learning
import umap                #Uniform Manifold Approximation and Projection (UMAP) is a dimensionality reduction technique
from sklearn.decomposition import PCA  #Principal component analysis (PCA). Linear dimensionality reduction using Singular Value    
from sklearn.manifold import TSNE #t-SNE [1] is a tool to visualize high-dimensional data
from itertools import product   #producte cartesià

import obtain_data   #per importar dades d'altres fitxers

#from astroML.plotting import setup_text_plots
#setup_text_plots(fontsize=16, usetex=True)




### Preparing the sample (mostra de RedClumps)
#### The GALAH DR3 RC sample
The GALAH DR3 red-clump sample is a selection of RC stars from the GALAH DR3 catalogue. I have put together the possibly interesting portions of the catalogs and cut a sample of red clump (RC) stars using topcat: (teff > 4500 && teff < 5100 && logg > 2.3 && logg < 2.55 && is_redclump_bstep > 0.5). This sample contains 37,417 stars.

In [10]:
import importlib   #package per importar coses a python
importlib.reload(obtain_data)

galah_rc = obtain_data.galah_dr3_rc()   

### Prepare the input arrays for XD

In [11]:
galah_rc.get_ndimspace(feh=True, norm="stdev")

X = galah_rc.X            #(10941, 24)
Xerr = galah_rc.Xerr1         #(10941, 24)
Xcov = np.zeros(Xerr.shape + Xerr.shape[-1:])
Xcov[:, range(Xerr.shape[1]), range(Xerr.shape[1])] = Xerr ** 2    #(10941, 24, 24)

Cal calcular una matriu W (mixture matrix)???  no pq errors d'un element no influeixen en errors d'un altre, en una mateixa estrella

In [ ]:
for n_clusters in range(5, 31, 5):
    
    @pickle_results('XD_'+str(n_clusters)+'clusters.pkl')
    def compute_XD(n_clusters=n_clusters, rseed=0, max_iter=100, verbose=True):
        np.random.seed(rseed)
        clf = XDGMM(n_clusters, max_iter=max_iter, tol=1E-5, verbose=verbose)
        clf.fit(X, Xcov) 
        return clf
    # Fit the model on training set
    clf = compute_XD(n_clusters)  #la primera vegada que es crida la funció ho guarda a l'arxiu pkl

@pickle_results: computing results and saving to 'XD_5clusters.pkl'


/Applications/anaconda3/lib/python3.8/site-packages/sklearn/mixture/_base.py:265: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn('Initialization %d did not converge. '


1: log(L) = 2.6007e+05
    (10 sec)
2: log(L) = 2.6283e+05
    (10 sec)
3: log(L) = 2.6454e+05
    (10 sec)
4: log(L) = 2.6572e+05
    (10 sec)
5: log(L) = 2.666e+05
    (10 sec)
6: log(L) = 2.6728e+05
    (10 sec)
7: log(L) = 2.6782e+05
    (12 sec)
8: log(L) = 2.6826e+05
    (11 sec)
9: log(L) = 2.6863e+05
    (10 sec)
10: log(L) = 2.6895e+05
    (10 sec)
11: log(L) = 2.6922e+05
    (11 sec)
12: log(L) = 2.6946e+05
    (11 sec)
13: log(L) = 2.6967e+05
    (11 sec)
14: log(L) = 2.6986e+05
    (11 sec)
15: log(L) = 2.7002e+05
    (11 sec)
16: log(L) = 2.7017e+05
    (10 sec)
17: log(L) = 2.703e+05
    (10 sec)
18: log(L) = 2.7042e+05
    (10 sec)
19: log(L) = 2.7052e+05
    (10 sec)
20: log(L) = 2.7062e+05
    (10 sec)
21: log(L) = 2.7071e+05
    (10 sec)
22: log(L) = 2.7079e+05
    (10 sec)
23: log(L) = 2.7086e+05
    (10 sec)
24: log(L) = 2.7093e+05
    (10 sec)
25: log(L) = 2.71e+05
    (10 sec)
26: log(L) = 2.7105e+05
    (10 sec)
27: log(L) = 2.7111e+05
    (10 sec)
28: log(L) = 2

/Applications/anaconda3/lib/python3.8/site-packages/sklearn/mixture/_base.py:265: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn('Initialization %d did not converge. '


1: log(L) = 2.6389e+05
    (52 sec)
2: log(L) = 2.6672e+05
    (44 sec)
3: log(L) = 2.6847e+05
    (41 sec)
4: log(L) = 2.6973e+05
    (41 sec)
5: log(L) = 2.7069e+05
    (41 sec)
6: log(L) = 2.7143e+05
    (41 sec)
7: log(L) = 2.7202e+05
    (48 sec)
8: log(L) = 2.725e+05
    (43 sec)
9: log(L) = 2.729e+05
    (52 sec)
10: log(L) = 2.7324e+05
    (45 sec)
11: log(L) = 2.7354e+05
    (53 sec)
12: log(L) = 2.7379e+05
    (52 sec)
13: log(L) = 2.7402e+05
    (56 sec)
14: log(L) = 2.7423e+05
    (58 sec)
15: log(L) = 2.7441e+05
    (55 sec)
16: log(L) = 2.7457e+05
    (66 sec)
17: log(L) = 2.7472e+05
    (58 sec)
18: log(L) = 2.7486e+05
    (61 sec)
19: log(L) = 2.7499e+05
    (59 sec)
20: log(L) = 2.7511e+05
    (60 sec)
21: log(L) = 2.7522e+05
    (60 sec)
22: log(L) = 2.7532e+05
    (58 sec)
23: log(L) = 2.7541e+05
    (54 sec)
24: log(L) = 2.7549e+05
    (56 sec)
25: log(L) = 2.7557e+05
    (57 sec)
26: log(L) = 2.7565e+05
    (55 sec)
27: log(L) = 2.7572e+05
    (46 sec)
28: log(L) =

...

/Applications/anaconda3/lib/python3.8/site-packages/sklearn/mixture/_base.py:265: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn('Initialization %d did not converge. '


1: log(L) = 2.6808e+05
    (2.4e+02 sec)
2: log(L) = 2.7098e+05
    (2.3e+02 sec)
3: log(L) = 2.7279e+05
    (2.4e+02 sec)
4: log(L) = 2.7406e+05
    (2.6e+02 sec)
5: log(L) = 2.75e+05
    (2.4e+02 sec)
6: log(L) = 2.7573e+05
    (2.3e+02 sec)
7: log(L) = 2.7632e+05
    (2.3e+02 sec)
8: log(L) = 2.7679e+05
    (2.4e+02 sec)
9: log(L) = 2.7719e+05
    (2.4e+02 sec)
10: log(L) = 2.7753e+05
    (2.5e+02 sec)
11: log(L) = 2.7782e+05
    (2.3e+02 sec)
12: log(L) = 2.7807e+05
    (2.5e+02 sec)
13: log(L) = 2.7829e+05
    (2.5e+02 sec)
14: log(L) = 2.7849e+05
    (2.5e+02 sec)
15: log(L) = 2.7866e+05
    (2.3e+02 sec)
16: log(L) = 2.7881e+05
    (2.4e+02 sec)
17: log(L) = 2.7895e+05
    (2.5e+02 sec)
18: log(L) = 2.7908e+05
    (2.4e+02 sec)
19: log(L) = 2.792e+05
    (2.6e+02 sec)
20: log(L) = 2.7931e+05
    (2.5e+02 sec)
21: log(L) = 2.794e+05
    (2.4e+02 sec)
22: log(L) = 2.7949e+05
    (2.5e+02 sec)
23: log(L) = 2.7958e+05
    (2.5e+02 sec)
24: log(L) = 2.7966e+05
    (2.4e+02 sec)
25: l

/Applications/anaconda3/lib/python3.8/site-packages/sklearn/mixture/_base.py:265: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn('Initialization %d did not converge. '


1: log(L) = 2.688e+05
    (3.1e+02 sec)
2: log(L) = 2.7171e+05
    (3e+02 sec)
3: log(L) = 2.7352e+05
    (3.1e+02 sec)
4: log(L) = 2.7479e+05
    (3.2e+02 sec)
5: log(L) = 2.7574e+05
    (3.3e+02 sec)
6: log(L) = 2.7648e+05
    (3.2e+02 sec)
7: log(L) = 2.7708e+05
    (3.2e+02 sec)
8: log(L) = 2.7757e+05
    (3.2e+02 sec)
9: log(L) = 2.7798e+05
    (3.2e+02 sec)
10: log(L) = 2.7833e+05
    (3.1e+02 sec)
11: log(L) = 2.7864e+05
    (3e+02 sec)
12: log(L) = 2.7891e+05
    (3e+02 sec)
13: log(L) = 2.7914e+05
    (3.1e+02 sec)
14: log(L) = 2.7935e+05
    (3.1e+02 sec)
15: log(L) = 2.7954e+05
    (3.1e+02 sec)
16: log(L) = 2.7971e+05
    (3e+02 sec)
17: log(L) = 2.7986e+05
    (3.1e+02 sec)
18: log(L) = 2.8e+05
    (3e+02 sec)
19: log(L) = 2.8013e+05
    (3.1e+02 sec)
20: log(L) = 2.8024e+05
    (3.1e+02 sec)
21: log(L) = 2.8035e+05
    (3e+02 sec)
22: log(L) = 2.8045e+05
    (3.1e+02 sec)
23: log(L) = 2.8054e+05
    (3e+02 sec)
24: log(L) = 2.8062e+05
    (3.1e+02 sec)
25: log(L) = 2.807e

/Applications/anaconda3/lib/python3.8/site-packages/sklearn/mixture/_base.py:265: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn('Initialization %d did not converge. '


In [ ]:
np.random.seed(42)

#import corner 
#figure = corner.corner(np.array([galah_rc.data["R_Rzphi"],galah_rc.data["phi_Rzphi"], 
#                                 galah_rc.data["z_Rzphi"]]).T, 
#                       weights=None, bins=30, 
#                       range=[[4,12],[-.5,0.5],[-2,2]],
#                       labels=[r"$R_{\rm Gal}$ [kpc]", r"$\varphi$", 
#                               r"$Z_{\rm Gal}$ [kpc]"],
#                       quantiles=[0.16, 0.5, 0.84],plot_datapoints=False,
#                       show_titles=False, label_kwargs={"fontsize": 20})
#ç 
 
     - plt.savefig("../im/corner_rphiz.png")


for n_clusters in range(5, 31, 5):
    
    clf = compute_XD(n_clusters)
    # Fit and sample from the underlying distribution
    X_sample = clf.sample(X.shape[0])   #XD resampling X
    
    # plot the results
    fig = plt.figure(figsize=(10, 7.5))
    fig.subplots_adjust(left=0.12, right=0.95,
                        bottom=0.1, top=0.95,
                        wspace=0.02, hspace=0.02)

    # only plot 1/10 of the stars for clarity
    #standard
    ax1 = fig.add_subplot(221)
    ax1.scatter(X[::10, 0], X[::10, 1], s=9, lw=0, c='k')   #standard X
    #XDeconvolution
    ax2 = fig.add_subplot(222)
    ax2.scatter(X_sample[::10, 0], X_sample[::10, 1], s=9, lw=0, c='k')  #XD resampling X
    #elipses
    ax3 = fig.add_subplot(223)
    for i in range(clf.n_components):
        draw_ellipse(clf.mu[i, 0:2], clf.V[i, 0:2, 0:2], scales=[2],
                     ec='k', fc='gray', alpha=0.2, ax=ax3)

    titles = ["Standard Distribution","Extreme Deconvolution\n resampling",
              "Extreme Deconvolution\n  cluster locations"]
    ax = [ax1, ax2, ax3]

    for i in range(3):
        ax[i].set_xlim(-5, 5)
        ax[i].set_ylim(-5, 5)

        ax[i].xaxis.set_major_locator(plt.MultipleLocator(1))
        ax[i].yaxis.set_major_locator(plt.MultipleLocator(1))

        ax[i].text(0.05, 0.95, titles[i],
                   ha='left', va='top', transform=ax[i].transAxes)

        if i in (0, 1):
            ax[i].xaxis.set_major_formatter(plt.NullFormatter())
        else:
            ax[i].set_xlabel('$g-r$')

correr per 24 dimensions guardant a pickle (així no cal esperar cada vegada), plotejar primer unes 5.
fer for per n_clusters = 5, 10, 15.., 30.